<h1> Collect first 200 data for each keyword</h1>

The original ipython notebook that was used for the data collection was lost; however, you can see the code that was used in the following cells. Approximetely, 200 vidoes per keyword per year was collected

**Let's use the following code:**

In [ ]:
#API keys:
FIRST_API = '***'
SECOND_API = '***'
THIRD_API = '***'
FOURTH_API = '***'

In [ ]:
# getting variables ready
from googleapiclient.discovery import build
import pandas as pd
import time

API_KEY = SECOND_API
youtube = build('youtube', 'v3', developerKey = API_KEY)
SEARCH_TERMS = ["biohacking", "longevity", "healthy aging", "biohacking", "health rejuvenation", "anti-aging", "nootropics"]

In [ ]:
def get_all_videos(query, max_pages=4):
    all_results = []
    next_page_token = None

    for _ in range(max_pages):
        search_response = youtube.search().list(
            q=query,
            part='snippet',
            publishedAfter="2024-01-01T00:00:00Z",
            publishedBefore="2025-01-01T00:00:00Z",
            maxResults=50,
            type='video',
            pageToken=next_page_token
        ).execute()

        all_results.extend(search_response['items'])

        next_page_token = search_response.get('nextPageToken')
        if not next_page_token:
            break

    return all_results

In [ ]:
video_data = []

for term in SEARCH_TERMS:
    search_response = get_all_videos(term)
    
    for item in search_response:
        video_id = item['id']['videoId']
        title = item['snippet']['title']
        description = item['snippet']['description']
        published_at = item['snippet']['publishedAt']
        channel_title = item['snippet']['channelTitle']


        video_response = youtube.videos().list(
            part='statistics,snippet',
            id=video_id
        ).execute()

        stats = video_response['items'][0]['statistics']
        snippet = video_response['items'][0]['snippet']

        # collect the top-level comments
        try:
            comment_response = youtube.commentThreads().list(
                part='snippet',
                videoId=video_id,
                maxResults=5  
            ).execute()

            comments = [c['snippet']['topLevelComment']['snippet']['textDisplay']
                        for c in comment_response.get('items', [])]
            comment_text = " || ".join(comments)

        except:
            comment_text = ""

        

        video_data.append({
            "source": "YouTube",
            "video_id": video_id,
            "video_url": f"https://www.youtube.com/watch?v={video_id}",
            "title": title,
            "description": description,
            "date": published_at,
            "channel": channel_title,
            "comments": comment_text,
            "views": stats.get('viewCount'),
            "likes": stats.get('likeCount'),
            "comments_count": stats.get('commentCount'),
            "keywords": term,
            "country": snippet.get('defaultAudioLanguage', 'unknown')
        })
    
        time.sleep(1)  # Avoid hitting API rate limits


df = pd.DataFrame(video_data)
df.to_csv("first_200_2024_biohacking.csv", index=False)
print("Done! Collected", len(df), "videos.")

<h1> Cleaning + EDA </h1>

We have collected 100 vidoes from each year; however, starting from the year 2019, we increased the number of videos to 150. In below, you can see the data cleaning process and some basic analysis regarding the data. 

<h2> Basic Data cleaning </h2>

In [112]:
import pandas as pd

df_list = []
for i in range(1, 17):
    globals()[f"df_{i}"] = pd.read_csv(f"first_100_20{i+9}.csv")
    df_list.append(globals()[f"df_{i}"])

In [114]:
df = pd.concat(df_list, ignore_index=True)

<h3> Duplicated values and data column</h3>

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date_only'] = df['date'].dt.date
df.drop(columns=['date'], inplace=True)
df.rename(columns={"date_only": "date"}, inplace=True)
df = df.drop_duplicates(keep='first')
df=df.sort_values('date').drop_duplicates(subset='video_id', keep='last')

<h3>Null values</h3>

In [85]:
df['description'] = df['description'].fillna('')
df['comments']=df['comments'].fillna('')
df['comments_count']=df['comments_count'].fillna(0)
df = df[~df['views'].isnull()] 

In [90]:
df['likes_missing'] = df['likes'].isnull()

In [96]:
df['likes'] = df['likes'].fillna(0)

In [102]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

<h4>Basic engagement analysis</h4>

Let's create the engagement column for each video which represents the how the users are engaged. It will be the sum of likes and comment counts

In [121]:
df['engagement']=df['likes'] + df['comments_count']

**Engagement rate**

Let's create the column called 'engagement_rate'. This columsn will represent the percentage of the people who have actively engaged with the video. This can give give us the insights such as which topics (keywords) have created more discussions. The rate will be engagement/views

A video with 10,000 views and 200 total interactions (2%) may be more engaging than one with 1M views and 5,000 interactions (0.5%).

In [129]:
df['engagement_rate'] = df['engagement'] / df['views']
df.loc[df['views'] == 0, 'engagement_rate'] = 0

let's save the data for the later use

In [ ]:
df.to_csv("clean_data.csv", index=False)